# Phase 3 v5 — Forecasting, Scenario Simulation, and Model Validation with the Final Phase 2 Lasso Backbone

This notebook carries forward the **final Phase 2 model** into Phase 3 scenario forecasting.

## Backbone used in this version
- **Final structural model:** tuned **Lasso**
- **Phase 2 test R²:** **0.8678**
- **Phase 2 test RMSE:** **0.5468**
- **Split design:** chronological **80/20** train-test split with validation inside the training block

## Main design
- **Baseline structural forecast:** uses the finalized Phase 2 lagged / rolling feature set and the same Lasso model family selected in Phase 2.
- **Scenario layer:** applies hurricane and earthquake shocks through the same engineered feature space used by the Lasso backbone, plus a stylized event-year overlay.
- **Calibration / stress testing:** includes a Maria-like backtest path for island and regional trajectories.
- **Scales:** municipal, regional, and island outputs through 2030.
- **Charts:** expanded comparison, ladder, impact, and backtest visualizations.

The goal is to ensure that Phase 3 scenario testing remains consistent with the **actual winning Phase 2 model**, rather than a different tree-based backbone.


In [1]:
import os, json, warnings, math, re
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import Lasso
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

warnings.filterwarnings("ignore")

np.random.seed(42)


In [2]:

# ----------------------------
# Output folder
# ----------------------------
OUTPUT_DIR = Path("phase3_outputs_v5")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def save_csv(df, name):
    path = OUTPUT_DIR / name
    df.to_csv(path, index=False)
    print(f"Saved CSV: {path}")
    return str(path)

def save_json(obj, name):
    path = OUTPUT_DIR / name
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)
    print(f"Saved JSON: {path}")
    return str(path)

def save_fig(fig, name, dpi=150):
    path = OUTPUT_DIR / name
    fig.savefig(path, bbox_inches="tight", dpi=dpi)
    print(f"Saved FIG: {path}")
    plt.close(fig)
    return str(path)

saved_files = []
chart_index_records = []


In [3]:

# ----------------------------
# File paths
# ----------------------------
preferred_files = [
    "processed_puerto_rico_data_rebuild.csv",
    "processed_puerto_rico_data_enriched.csv",
]

data_path = None
for p in preferred_files:
    if Path(p).exists():
        data_path = Path(p)
        break

if data_path is None:
    raise FileNotFoundError(
        "Could not find processed_puerto_rico_data_rebuild.csv or processed_puerto_rico_data_enriched.csv in the current directory."
    )

print("Using data file:", data_path)
df = pd.read_csv(data_path)
print(df.shape)
df.head()


Using data file: processed_puerto_rico_data_rebuild.csv
(1170, 160)


,year,municipio,total_population,total_population_moe,total_population_16plus,total_population_16plus_moe,total_housing_units,total_housing_units_moe,median_income_nominal,median_income_real,...,wind_3yr_x_svi_current,seismic_3yr_x_svi_current,years_since_hurricane,years_since_earthquake,years_since_hurricane_capped_5,years_since_earthquake_capped_5,target_pop_change_1y_winsor_1_99,target_pop_change_2y_avg_winsor_1_99,target_pop_change_3y_avg_winsor_1_99,municipality_id
0,2010,Adjuntas,19541.0,0,14834,99,6548,133,11983.0,16743.095245,...,0.063443,3055.178423,0,0,0,0,-0.138171,-0.212600,-0.354358,0
1,2011,Adjuntas,19514.0,0,15000,107,6834,161,12975.0,17577.288950,...,0.550670,4292.995237,0,0,0,0,-0.286973,-0.462276,-0.559995,0
2,2012,Adjuntas,19458.0,0,15081,80,7037,114,13095.0,17379.542368,...,0.550670,4862.171141,1,0,1,0,-0.637270,-0.696226,-0.857017,0
3,2013,Adjuntas,19334.0,0,15087,79,7298,140,11528.0,15078.786201,...,0.645874,2495.720963,0,0,0,0,-0.755146,-0.966708,-0.999579,0
4,2014,Adjuntas,19188.0,0,15080,88,7514,136,10550.0,13580.165846,...,0.265284,6766.905850,0,0,0,0,-1.177819,-1.121570,-1.165288,0


In [4]:

# ----------------------------
# Region map
# ----------------------------
region_map = {
    # Metro
    "San Juan": "Metro", "Bayamón": "Metro", "Carolina": "Metro", "Cataño": "Metro",
    "Guaynabo": "Metro", "Toa Alta": "Metro", "Toa Baja": "Metro", "Trujillo Alto": "Metro",
    # North
    "Arecibo": "North", "Barceloneta": "North", "Camuy": "North", "Dorado": "North",
    "Florida": "North", "Hatillo": "North", "Manatí": "North", "Quebradillas": "North",
    "Vega Alta": "North", "Vega Baja": "North",
    # South
    "Arroyo": "South", "Coamo": "South", "Guayama": "South", "Guayanilla": "South",
    "Juana Díaz": "South", "Patillas": "South", "Peñuelas": "South", "Ponce": "South",
    "Salinas": "South", "Santa Isabel": "South", "Villalba": "South", "Yauco": "South",
    # West
    "Aguada": "West", "Aguadilla": "West", "Añasco": "West", "Cabo Rojo": "West",
    "Guánica": "West", "Hormigueros": "West", "Isabela": "West", "Lajas": "West",
    "Las Marías": "West", "Maricao": "West", "Mayagüez": "West", "Moca": "West",
    "Rincón": "West", "Sabana Grande": "West", "San Germán": "West", "San Sebastián": "West",
    # East
    "Canóvanas": "East", "Ceiba": "East", "Fajardo": "East", "Humacao": "East",
    "Juncos": "East", "Las Piedras": "East", "Loíza": "East", "Luquillo": "East",
    "Maunabo": "East", "Naguabo": "East", "Río Grande": "East", "San Lorenzo": "East",
    "Yabucoa": "East", "Caguas": "East", "Gurabo": "East", "Culebra": "East", "Vieques": "East",
    # Central Mountains
    "Adjuntas": "Central Mountains", "Aguas Buenas": "Central Mountains", "Aibonito": "Central Mountains",
    "Barranquitas": "Central Mountains", "Cayey": "Central Mountains", "Ciales": "Central Mountains",
    "Cidra": "Central Mountains", "Comerío": "Central Mountains", "Corozal": "Central Mountains",
    "Jayuya": "Central Mountains", "Lares": "Central Mountains", "Morovis": "Central Mountains",
    "Naranjito": "Central Mountains", "Orocovis": "Central Mountains", "Utuado": "Central Mountains",
}


In [5]:

# ----------------------------
# Basic cleanup / canonical columns
# ----------------------------
rename_candidates = {
    "municipality": "municipio",
    "Municipio": "municipio",
    "Municipality": "municipio",
    "Region": "region",
    "Year": "year",
    "total_pop": "total_population",
    "population": "total_population",
}

for old, new in rename_candidates.items():
    if old in df.columns and new not in df.columns:
        df = df.rename(columns={old: new})

if "municipio" not in df.columns:
    raise KeyError("Expected a 'municipio' column.")
if "year" not in df.columns:
    raise KeyError("Expected a 'year' column.")
if "total_population" not in df.columns:
    # fallback: choose a likely population column
    pop_candidates = [c for c in df.columns if "population" in c.lower()]
    if not pop_candidates:
        raise KeyError("Could not identify total population column.")
    df["total_population"] = pd.to_numeric(df[pop_candidates[0]], errors="coerce")

df["municipio"] = df["municipio"].astype(str).str.strip()
if "region" not in df.columns:
    df["region"] = df["municipio"].map(region_map)
else:
    df["region"] = df["region"].fillna(df["municipio"].map(region_map))

df["year"] = pd.to_numeric(df["year"], errors="coerce").astype(int)
df["total_population"] = pd.to_numeric(df["total_population"], errors="coerce")
df = df.sort_values(["municipio", "year"]).reset_index(drop=True)

print(df[["municipio","region","year","total_population"]].head())
print(df.shape)


  municipio             region  year  total_population
0  Adjuntas  Central Mountains  2010           19541.0
1  Adjuntas  Central Mountains  2011           19514.0
2  Adjuntas  Central Mountains  2012           19458.0
3  Adjuntas  Central Mountains  2013           19334.0
4  Adjuntas  Central Mountains  2014           19188.0
(1170, 161)


In [6]:

# ----------------------------
# Build the finalized Phase 2 feature space on the Phase 3 dataframe
# ----------------------------
g = df.groupby("municipio")["total_population"]

if "target_pop_change_1y" not in df.columns:
    df["target_pop_change_1y"] = ((g.shift(-1) / df["total_population"]) - 1) * 100

if "target_pop_change_3y_avg" not in df.columns:
    df["target_pop_change_3y_avg"] = (((g.shift(-3) / df["total_population"]) ** (1/3)) - 1) * 100

if "lag_pop_change_1" not in df.columns:
    df["lag_pop_change_1"] = g.pct_change() * 100
if "lag_pop_change_2" not in df.columns:
    df["lag_pop_change_2"] = df.groupby("municipio")["lag_pop_change_1"].shift(1)

for col in [
    "years_since_hurricane", "years_since_earthquake", "post_maria", "post_earthquake_2020",
    "wind_3yr_sum", "seismic_3yr_sum", "wind_3yr_x_svi", "seismic_3yr_x_svi",
    "svi_pca_1", "income_growth", "establishment_growth", "lag_total_crime_rate"
]:
    if col not in df.columns:
        df[col] = 0.0

df["region"] = df["region"].fillna("Unknown")

def first_existing(candidates, frame=df):
    for c in candidates:
        if c in frame.columns:
            return c
    return None

resolved = {
    "population": first_existing(["total_population", "population"]),
    "target": first_existing(["target_pop_change_3y_avg", "target_pop_change_2y_avg", "target_pop_change_1y"]),
    "svi": first_existing(["svi_pca_1", "baseline_svi_pca_1", "baseline_vulnerability_index", "vulnerability_index"]),
    "wind": first_existing(["wind_3yr_sum", "lag_wind_exposure_score_100km", "max_wind_knots"]),
    "seismic": first_existing(["seismic_3yr_sum", "lag_seismic_exposure_score_50km", "max_seismic_mag"]),
    "wind_x_svi": first_existing(["wind_3yr_x_svi", "hurricane_x_svi"]),
    "seis_x_svi": first_existing(["seismic_3yr_x_svi", "earthquake_x_svi"]),
    "income_growth": first_existing(["income_growth", "income_growth_3yr_avg"]),
    "est_growth": first_existing(["establishment_growth", "lag_establishment_growth", "establishment_count"]),
    "crime": first_existing(["lag_total_crime_rate", "total_crime_rate"]),
}

target_col = resolved["target"]
if target_col is None:
    raise ValueError("No target column found.")

lag_source_candidates = [
    resolved["target"],
    resolved["svi"],
    resolved["wind"],
    resolved["seismic"],
    resolved["income_growth"],
    resolved["est_growth"],
    resolved["crime"],
    resolved["population"],
]
lag_source_candidates = [c for c in lag_source_candidates if c is not None and c in df.columns]
grouped = df.groupby("municipio", group_keys=False)

for col in lag_source_candidates:
    safe = re.sub(r"[^0-9a-zA-Z_]+", "_", str(col)).strip("_").lower()

    lag1 = f"{safe}_lag1"
    if lag1 not in df.columns:
        df[lag1] = grouped[col].shift(1)

    lag2 = f"{safe}_lag2"
    if lag2 not in df.columns:
        df[lag2] = grouped[col].shift(2)

    roll3 = f"{safe}_roll3_mean"
    if roll3 not in df.columns:
        df[roll3] = (
            grouped[col]
            .apply(lambda s: s.shift(1).rolling(3, min_periods=1).mean())
            .reset_index(level=0, drop=True)
        )

    yoy = f"{safe}_yoy_change"
    if yoy not in df.columns:
        df[yoy] = grouped[col].diff(1)

base_feature_candidates = [
    "lag_pop_change_1",
    "lag_pop_change_2",
    resolved["population"],
    resolved["svi"],
    resolved["wind"],
    resolved["seismic"],
    resolved["wind_x_svi"],
    resolved["seis_x_svi"],
    resolved["income_growth"],
    resolved["est_growth"],
    resolved["crime"],
    "year",
]

generated_feature_candidates = []
for col in lag_source_candidates:
    safe = re.sub(r"[^0-9a-zA-Z_]+", "_", str(col)).strip("_").lower()
    generated_feature_candidates.extend([
        f"{safe}_lag1",
        f"{safe}_lag2",
        f"{safe}_roll3_mean",
        f"{safe}_yoy_change",
    ])

PHASE2_FEATURES = [c for c in base_feature_candidates + generated_feature_candidates if c is not None and c in df.columns]
PHASE2_FEATURES = list(dict.fromkeys(PHASE2_FEATURES))

print("Resolved core columns:", resolved)
print("Phase 2 Lasso feature count:", len(PHASE2_FEATURES))


Resolved core columns: {'population': 'total_population', 'target': 'target_pop_change_3y_avg', 'svi': 'svi_pca_1', 'wind': 'wind_3yr_sum', 'seismic': 'seismic_3yr_sum', 'wind_x_svi': 'wind_3yr_x_svi', 'seis_x_svi': 'seismic_3yr_x_svi', 'income_growth': 'income_growth', 'est_growth': 'establishment_growth', 'crime': 'lag_total_crime_rate'}
Phase 2 Lasso feature count: 44


In [7]:

# ----------------------------
# Final Phase 2 Lasso feature set availability check
# ----------------------------
missing = [c for c in PHASE2_FEATURES if c not in df.columns]
if missing:
    raise KeyError(f"Missing finalized Phase 2 features: {missing}")

print("Finalized Phase 2 Lasso backbone features available.")
print(PHASE2_FEATURES)


Finalized Phase 2 Lasso backbone features available.
['lag_pop_change_1', 'lag_pop_change_2', 'total_population', 'svi_pca_1', 'wind_3yr_sum', 'seismic_3yr_sum', 'wind_3yr_x_svi', 'seismic_3yr_x_svi', 'income_growth', 'establishment_growth', 'lag_total_crime_rate', 'year', 'target_pop_change_3y_avg_lag1', 'target_pop_change_3y_avg_lag2', 'target_pop_change_3y_avg_roll3_mean', 'target_pop_change_3y_avg_yoy_change', 'svi_pca_1_lag1', 'svi_pca_1_lag2', 'svi_pca_1_roll3_mean', 'svi_pca_1_yoy_change', 'wind_3yr_sum_lag1', 'wind_3yr_sum_lag2', 'wind_3yr_sum_roll3_mean', 'wind_3yr_sum_yoy_change', 'seismic_3yr_sum_lag1', 'seismic_3yr_sum_lag2', 'seismic_3yr_sum_roll3_mean', 'seismic_3yr_sum_yoy_change', 'income_growth_lag1', 'income_growth_lag2', 'income_growth_roll3_mean', 'income_growth_yoy_change', 'establishment_growth_lag1', 'establishment_growth_lag2', 'establishment_growth_roll3_mean', 'establishment_growth_yoy_change', 'lag_total_crime_rate_lag1', 'lag_total_crime_rate_lag2', 'lag_tot

In [8]:

# ----------------------------
# Load or reconstruct the final Phase 2 Lasso backbone
# Target: 3-year average forward population change
# ----------------------------
model_df = df.copy()
model_df = model_df.sort_values(["municipio", "year"]).reset_index(drop=True)
model_df = model_df.loc[:, ~model_df.columns.duplicated()].copy()

valid = model_df[target_col].notna()
model_df = model_df.loc[valid].copy()

feature_cols = [c for c in PHASE2_FEATURES if c in model_df.columns and model_df[c].notna().any()]
cat_cols = [c for c in ["municipio", "region"] if c in model_df.columns]

available_years = sorted(pd.to_numeric(model_df["year"], errors="coerce").dropna().astype(int).unique())
print("Available target years:", available_years)

if 2019 in available_years and any(y >= 2020 for y in available_years):
    outer_train_years = [y for y in available_years if y <= 2019]
    train_years = [y for y in available_years if y <= 2018]
    val_year = 2019
    test_years = [y for y in available_years if y >= 2020]
else:
    split_idx = max(1, int(np.floor(len(available_years) * 0.8)))
    outer_train_years = available_years[:split_idx]
    test_years = available_years[split_idx:]
    val_year = outer_train_years[-1]
    train_years = outer_train_years[:-1]

print("Pre-validation training block:", outer_train_years)
print("Final train years:", train_years)
print("Validation year:", val_year)
print("Test years:", test_years)

train_df = model_df[model_df["year"].isin(train_years)].copy()
val_df = model_df[model_df["year"] == val_year].copy()
test_df = model_df[model_df["year"].isin(test_years)].copy()

X_train = train_df[feature_cols + cat_cols].copy()
y_train = train_df[target_col].copy()
X_val = val_df[feature_cols + cat_cols].copy()
y_val = val_df[target_col].copy()
X_test = test_df[feature_cols + cat_cols].copy()
y_test = test_df[target_col].copy()

def metric_block(y_true, y_pred):
    return {
        "r2": float(r2_score(y_true, y_pred)),
        "rmse": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "mae": float(mean_absolute_error(y_true, y_pred)),
    }

def find_first_existing(paths):
    for p in paths:
        p = Path(p)
        if p.exists():
            return p.resolve()
    return None

phase2_dir_candidates = [
    Path("phase2_outputs_v7"), Path("phase2_outputs_v6"), Path("phase2_outputs_v5"), Path("phase2_outputs_v7"),
    OUTPUT_DIR.parent / "phase2_outputs_v7",
    OUTPUT_DIR.parent / "phase2_outputs_v6",
    OUTPUT_DIR.parent / "phase2_outputs_v5",
    OUTPUT_DIR.parent / "phase2_outputs_v7",
]

pipeline_file_candidates = []
summary_file_candidates = []
for d in phase2_dir_candidates:
    pipeline_file_candidates.extend([
        d / "phase2_v7_best_pipeline.joblib",
        d / "phase2_v6_best_pipeline.joblib",
        d / "phase2_v5_best_pipeline.joblib",
        d / "phase2_v4_best_pipeline.joblib",
    ])
    summary_file_candidates.extend([
        d / "phase2_v7_best_model_summary.json",
        d / "phase2_v6_best_model_summary.json",
        d / "phase2_v5_best_model_summary.json",
        d / "phase2_v4_best_model_summary.json",
    ])

phase2_pipeline_path = find_first_existing(pipeline_file_candidates)
reference_phase2_path = find_first_existing(summary_file_candidates)

phase2_pipeline_loaded = False
phase2_reference = None

if phase2_pipeline_path is not None:
    phase2_backbone_model = joblib.load(phase2_pipeline_path)
    phase2_pipeline_loaded = True
    print(f"Loaded fitted Phase 2 pipeline: {phase2_pipeline_path}")
else:
    print("No saved Phase 2 pipeline found; reconstructing the final Lasso backbone inside Phase 3.")
    num_cols = [c for c in feature_cols if c not in cat_cols]

    try:
        ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        ohe = OneHotEncoder(handle_unknown="ignore", sparse=False)

    preprocessor = ColumnTransformer(
        transformers=[
            ("num", Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
            ]), num_cols),
            ("cat", Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("ohe", ohe),
            ]), cat_cols),
        ],
        remainder="drop"
    )

    phase2_backbone_model = Pipeline([
        ("prep", preprocessor),
        ("model", Lasso(alpha=0.05, max_iter=50000, random_state=42))
    ])
    phase2_backbone_model.fit(X_train, y_train)

if reference_phase2_path is not None:
    with open(reference_phase2_path, "r", encoding="utf-8") as f:
        phase2_reference = json.load(f)
    print(f"Loaded Phase 2 reference summary: {reference_phase2_path}")
else:
    phase2_reference = {
        "model": "lasso",
        "val_r2": 0.9373345432852163,
        "test_r2": 0.8678244275012517,
        "test_rmse": 0.5467790560200507,
    }
    print("No saved Phase 2 summary found; using the confirmed final Phase 2 Lasso metrics as reference.")

val_pred = phase2_backbone_model.predict(X_val)
test_pred = phase2_backbone_model.predict(X_test)

historical_eval_df = pd.DataFrame([
    {"split": "validation", **metric_block(y_val, val_pred)},
    {"split": "test", **metric_block(y_test, test_pred)},
])

backbone_summary = {
    "model": "lasso",
    "target": target_col,
    "features": feature_cols + cat_cols,
    "n_train": int(len(train_df)),
    "n_val": int(len(val_df)),
    "n_test": int(len(test_df)),
    "expected_phase2_test_r2": 0.8678,
    "expected_phase2_test_rmse": 0.5468,
    "metrics": {
        "validation": metric_block(y_val, val_pred),
        "test": metric_block(y_test, test_pred),
    },
    "phase2_pipeline_loaded": phase2_pipeline_loaded,
    "reference_phase2_summary_loaded": bool(phase2_reference is not None),
}

if phase2_reference is not None:
    ref_val_r2 = phase2_reference.get("val_r2")
    ref_test_r2 = phase2_reference.get("test_r2")
    ref_test_rmse = phase2_reference.get("test_rmse")
    backbone_summary["phase2_reference_metrics"] = {
        "validation_r2": ref_val_r2,
        "test_r2": ref_test_r2,
        "test_rmse": ref_test_rmse,
    }
    backbone_summary["reproduction_gap"] = {
        "validation_r2_gap": None if ref_val_r2 is None else float(backbone_summary["metrics"]["validation"]["r2"] - ref_val_r2),
        "test_r2_gap": None if ref_test_r2 is None else float(backbone_summary["metrics"]["test"]["r2"] - ref_test_r2),
        "test_rmse_gap": None if ref_test_rmse is None else float(backbone_summary["metrics"]["test"]["rmse"] - ref_test_rmse),
    }

MODEL_YEAR_CAP = int(max(available_years))
print("MODEL_YEAR_CAP:", MODEL_YEAR_CAP)

backbone_summary


Available target years: [np.int64(2010), np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021)]
Pre-validation training block: [np.int64(2010), np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019)]
Final train years: [np.int64(2010), np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018)]
Validation year: 2019
Test years: [np.int64(2020), np.int64(2021)]
Loaded fitted Phase 2 pipeline: /Users/andreruiz/Documents/Capstone/CAPSTONE_DATA_SCIENCE_RIVERARUIZ/population_analysis copy/phase2_outputs_v6/phase2_v6_best_pipeline.joblib
Loaded Phase 2 reference summary: /Users/andreruiz/Documents/Capstone/CAPSTONE_DATA_SCIENCE_RIVERARUIZ/population_analysis copy/phase2_outputs_v6/phase2_v6_best_model_summary.json
MODEL

{'model': 'lasso',
 'target': 'target_pop_change_3y_avg',
 'features': ['lag_pop_change_1',
  'lag_pop_change_2',
  'total_population',
  'svi_pca_1',
  'wind_3yr_sum',
  'seismic_3yr_sum',
  'wind_3yr_x_svi',
  'seismic_3yr_x_svi',
  'income_growth',
  'establishment_growth',
  'lag_total_crime_rate',
  'year',
  'target_pop_change_3y_avg_lag1',
  'target_pop_change_3y_avg_lag2',
  'target_pop_change_3y_avg_roll3_mean',
  'target_pop_change_3y_avg_yoy_change',
  'svi_pca_1_lag1',
  'svi_pca_1_lag2',
  'svi_pca_1_roll3_mean',
  'svi_pca_1_yoy_change',
  'wind_3yr_sum_lag1',
  'wind_3yr_sum_lag2',
  'wind_3yr_sum_roll3_mean',
  'wind_3yr_sum_yoy_change',
  'seismic_3yr_sum_lag1',
  'seismic_3yr_sum_lag2',
  'seismic_3yr_sum_roll3_mean',
  'seismic_3yr_sum_yoy_change',
  'income_growth_lag1',
  'income_growth_lag2',
  'income_growth_roll3_mean',
  'income_growth_yoy_change',
  'establishment_growth_lag1',
  'establishment_growth_lag2',
  'establishment_growth_roll3_mean',
  'establishmen

In [9]:

saved_files.append(save_json(backbone_summary, "phase3_v5_backbone_model_summary.json"))


Saved JSON: phase3_outputs_v5/phase3_v5_backbone_model_summary.json


In [10]:
# Save clean backbone evaluation and optional comparison against the saved Phase 2 reference
saved_files.append(save_csv(historical_eval_df, "phase3_v5_clean_backbone_evaluation.csv"))
display(historical_eval_df)

if backbone_summary.get("reference_phase2_summary_loaded", False):
    comparison_rows = [
        {
            "split": "validation",
            "phase2_reference_r2": backbone_summary["phase2_reference_metrics"]["validation_r2"],
            "phase3_backbone_r2": backbone_summary["metrics"]["validation"]["r2"],
            "gap": backbone_summary["reproduction_gap"]["validation_r2_gap"],
        },
        {
            "split": "test",
            "phase2_reference_r2": backbone_summary["phase2_reference_metrics"]["test_r2"],
            "phase3_backbone_r2": backbone_summary["metrics"]["test"]["r2"],
            "gap": backbone_summary["reproduction_gap"]["test_r2_gap"],
        },
    ]
    backbone_comparison_df = pd.DataFrame(comparison_rows)
    saved_files.append(save_csv(backbone_comparison_df, "phase3_v5_backbone_vs_phase2_reference.csv"))
    display(backbone_comparison_df)
else:
    print("No saved Phase 2 summary found in phase2_outputs_v7; skipping backbone comparison.")


Saved CSV: phase3_outputs_v5/phase3_v5_clean_backbone_evaluation.csv


,split,r2,rmse,mae
0,validation,0.937335,0.444125,0.377442
1,test,0.867824,0.546779,0.487697


Saved CSV: phase3_outputs_v5/phase3_v5_backbone_vs_phase2_reference.csv


,split,phase2_reference_r2,phase3_backbone_r2,gap
0,validation,0.937335,0.937335,0.0
1,test,0.867824,0.867824,0.0


## Backbone-model reproducibility note

This notebook now uses the **final Phase 2 Lasso backbone**. When available, it loads the exact fitted Phase 2 pipeline artifact from a nearby Phase 2 output folder. If that artifact is missing, the notebook reconstructs the same model family directly inside Phase 3 using the finalized Phase 2 feature set and the confirmed tuned Lasso setting.

The confirmed Phase 2 benchmark for this backbone is:

- **Model:** Lasso  
- **Test R²:** **0.8678**  
- **Test RMSE:** **0.5468**

The backbone is evaluated first on a clean historical Phase 2-style feature matrix before any recursive forecasting or scenario overlays are applied. This keeps structural model validation separate from simulation logic.

During recursive forecasting, the model-year input is capped at the latest historically observed target year so future projections remain closer to the domain seen during training.


In [11]:

# ----------------------------
# Baseline annualized forecast builder
# We convert model's 3-year average pct change prediction into an annualized path.
# ----------------------------
history_df = df.copy()
latest_year = int(history_df["year"].max())
latest_rows = history_df.sort_values("year").groupby("municipio").tail(1).copy()
latest_rows = latest_rows.sort_values("municipio").reset_index(drop=True)

print("Latest year:", latest_year)
print("Latest rows:", latest_rows.shape)

# Municipio historical volatility used for uncertainty bands and shock heterogeneity
muni_vol = (
    model_df.groupby("municipio")["target_pop_change_1y"]
    .std()
    .fillna(model_df["target_pop_change_1y"].std())
    .to_dict()
)

global_vol = float(model_df["target_pop_change_1y"].std())
global_vol


Latest year: 2024
Latest rows: (78, 193)


1.9308983000926878

In [12]:

# ----------------------------
# Scenario definitions
# Hybrid design:
# 1) structural baseline path from Phase 2 backbone
# 2) explicit event-year / decay overlay calibrated for visible shock dynamics
# ----------------------------
HURRICANE_LADDER = {
    "baseline": {"kind": "none"},
    "hurricane_cat1": {"kind": "hurricane", "category": 1, "event_year": 2026},
    "hurricane_cat2": {"kind": "hurricane", "category": 2, "event_year": 2026},
    "hurricane_cat3": {"kind": "hurricane", "category": 3, "event_year": 2026},
    "hurricane_cat4": {"kind": "hurricane", "category": 4, "event_year": 2026},
    "hurricane_cat5_maria_like": {"kind": "hurricane", "category": 5, "event_year": 2026},
}

EARTHQUAKE_LADDER = {
    "earthquake_m6_0": {"kind": "earthquake", "magnitude": 6.0, "event_year": 2026},
    "earthquake_m6_5": {"kind": "earthquake", "magnitude": 6.5, "event_year": 2026},
    "earthquake_m6_8": {"kind": "earthquake", "magnitude": 6.8, "event_year": 2026},
    "earthquake_m7_2": {"kind": "earthquake", "magnitude": 7.2, "event_year": 2026},
    "earthquake_m7_5": {"kind": "earthquake", "magnitude": 7.5, "event_year": 2026},
}

scenario_definitions = {}
scenario_definitions.update(HURRICANE_LADDER)
scenario_definitions.update(EARTHQUAKE_LADDER)

saved_files.append(save_json(scenario_definitions, "phase3_v5_scenario_definitions.json"))
scenario_definitions


Saved JSON: phase3_outputs_v5/phase3_v5_scenario_definitions.json


{'baseline': {'kind': 'none'},
 'hurricane_cat1': {'kind': 'hurricane', 'category': 1, 'event_year': 2026},
 'hurricane_cat2': {'kind': 'hurricane', 'category': 2, 'event_year': 2026},
 'hurricane_cat3': {'kind': 'hurricane', 'category': 3, 'event_year': 2026},
 'hurricane_cat4': {'kind': 'hurricane', 'category': 4, 'event_year': 2026},
 'hurricane_cat5_maria_like': {'kind': 'hurricane',
  'category': 5,
  'event_year': 2026},
 'earthquake_m6_0': {'kind': 'earthquake',
  'magnitude': 6.0,
  'event_year': 2026},
 'earthquake_m6_5': {'kind': 'earthquake',
  'magnitude': 6.5,
  'event_year': 2026},
 'earthquake_m6_8': {'kind': 'earthquake',
  'magnitude': 6.8,
  'event_year': 2026},
 'earthquake_m7_2': {'kind': 'earthquake',
  'magnitude': 7.2,
  'event_year': 2026},
 'earthquake_m7_5': {'kind': 'earthquake',
  'magnitude': 7.5,
  'event_year': 2026}}

### Scenario-design note

The hurricane and earthquake scenarios in this notebook are **stylized scenario overlays**.  
They are designed to create transparent and interpretable stress tests on top of the structural baseline forecast. They should **not** be interpreted as estimated causal effects of specific future disasters.

In [13]:

# ----------------------------
# Shock calibration helpers
# These are explicit overlays on top of the Phase 2 baseline engine.
# The scale is in ANNUAL percent-population-change adjustments.
# ----------------------------
REGION_HURRICANE_MULT = {
    "East": 1.20,
    "South": 1.10,
    "Metro": 1.00,
    "Central Mountains": 1.05,
    "North": 0.95,
    "West": 1.00,
    "Unknown": 1.00,
}
REGION_EARTHQUAKE_MULT = {
    "South": 1.25,
    "West": 1.15,
    "Central Mountains": 1.05,
    "Metro": 0.95,
    "East": 0.95,
    "North": 0.90,
    "Unknown": 1.00,
}

def hurricane_base_annual_adjustment(category):
    # Explicit one-year shock followed by decay; tuned to produce visible response.
    return {1: -0.35, 2: -0.70, 3: -1.20, 4: -1.90, 5: -2.80}.get(category, -1.0)

def earthquake_base_annual_adjustment(magnitude):
    if magnitude < 6.0:
        return -0.20
    if magnitude < 6.5:
        return -0.45
    if magnitude < 6.8:
        return -0.75
    if magnitude < 7.2:
        return -1.05
    return -1.35

def shock_decay_profile(kind):
    # Event year, +1, +2 annual adjustment multipliers
    if kind == "hurricane":
        return [1.00, 0.60, 0.30]
    if kind == "earthquake":
        return [1.00, 0.45, 0.20]
    return [0.0, 0.0, 0.0]

def compute_maria_backtest_shock(event_year):
    # Maria-like backtest starts at 2017, stronger than Cat 4, with explicit recovery decay
    return {"kind": "hurricane", "category": 5, "event_year": event_year, "label": "maria_like_backtest"}

def apply_scenario_feature_overlay(base_row, year, scenario):
    row = base_row.copy()
    kind = scenario.get("kind", "none")
    event_year = scenario.get("event_year", 9999)

    # zero temporary shock inputs each year, then apply if in pulse window
    row["scenario_shock_event"] = 0.0
    row["scenario_shock_intensity"] = 0.0

    if kind == "none":
        return row, 0.0

    year_offset = year - event_year
    if year_offset < 0 or year_offset > 2:
        return row, 0.0

    decay = shock_decay_profile(kind)[year_offset]
    if kind == "hurricane":
        cat = scenario["category"]
        base_adj = hurricane_base_annual_adjustment(cat)
        region_mult = REGION_HURRICANE_MULT.get(row["region"], 1.0)
        svi_mult = 1.0 + max(0.0, float(row.get("svi_pca_1", 0.0))) * 0.08
        adj = base_adj * decay * region_mult * svi_mult

        row["wind_3yr_sum"] = float(row.get("wind_3yr_sum", 0.0)) + abs(base_adj) * 2.5 * decay
        row["wind_3yr_x_svi"] = float(row.get("wind_3yr_x_svi", 0.0)) + abs(base_adj) * max(0.0, float(row.get("svi_pca_1", 0.0))) * 2.0 * decay
        row["post_maria"] = 1.0
        row["years_since_hurricane"] = max(0, year - event_year)
        row["scenario_shock_event"] = 1.0 if year == event_year else 0.0
        row["scenario_shock_intensity"] = abs(base_adj) * decay
        return row, adj

    if kind == "earthquake":
        mag = scenario["magnitude"]
        base_adj = earthquake_base_annual_adjustment(mag)
        region_mult = REGION_EARTHQUAKE_MULT.get(row["region"], 1.0)
        svi_mult = 1.0 + max(0.0, float(row.get("svi_pca_1", 0.0))) * 0.06
        adj = base_adj * decay * region_mult * svi_mult

        row["seismic_3yr_sum"] = float(row.get("seismic_3yr_sum", 0.0)) + abs(base_adj) * 2.2 * decay
        row["seismic_3yr_x_svi"] = float(row.get("seismic_3yr_x_svi", 0.0)) + abs(base_adj) * max(0.0, float(row.get("svi_pca_1", 0.0))) * 1.7 * decay
        row["post_earthquake_2020"] = 1.0
        row["years_since_earthquake"] = max(0, year - event_year)
        row["scenario_shock_event"] = 1.0 if year == event_year else 0.0
        row["scenario_shock_intensity"] = abs(base_adj) * decay
        return row, adj

    return row, 0.0


In [14]:

# ----------------------------
# Forecast engine
# Baseline uses the validated final Phase 2 Lasso model.
# Scenario effects are injected into the same engineered feature space
# used by the Lasso backbone, plus a stylized annual overlay.
# ----------------------------
FORECAST_YEARS = list(range(latest_year + 1, 2031))
N_SIMS = 200
SAVE_DETAIL = False

holdout_actual = np.concatenate([y_val.values, y_test.values])
holdout_pred = np.concatenate([val_pred, test_pred])
phase2_holdout_resid = holdout_actual - holdout_pred
residual_std = float(np.std(phase2_holdout_resid))
print("Residual std from holdout residuals:", residual_std)

def predicted_annual_pct_to_decimal(pct_3y_avg_annualized):
    return pct_3y_avg_annualized / 100.0

muni_income_anchor = history_df.groupby("municipio")["income_growth"].median().to_dict() if "income_growth" in history_df.columns else {}
muni_est_anchor = history_df.groupby("municipio")["establishment_growth"].median().to_dict() if "establishment_growth" in history_df.columns else {}
muni_crime_anchor = history_df.groupby("municipio")["lag_total_crime_rate"].median().to_dict() if "lag_total_crime_rate" in history_df.columns else {}

global_income_anchor = float(history_df["income_growth"].median()) if "income_growth" in history_df.columns else 0.0
global_est_anchor = float(history_df["establishment_growth"].median()) if "establishment_growth" in history_df.columns else 0.0
global_crime_anchor = float(history_df["lag_total_crime_rate"].median()) if "lag_total_crime_rate" in history_df.columns else 0.0

lag_feature_bases = [c for c in lag_source_candidates if c in df.columns]

def _safe_name(col):
    return re.sub(r"[^0-9a-zA-Z_]+", "_", str(col)).strip("_").lower()

def refresh_recursive_features(sim_df, realized_pct=None):
    out = sim_df.copy()

    if realized_pct is not None and "target_pop_change_3y_avg" in out.columns:
        out["target_pop_change_3y_avg"] = np.asarray(realized_pct, dtype=float)

    for base_col in lag_feature_bases:
        if base_col not in out.columns:
            continue
        safe = _safe_name(base_col)
        lag1 = f"{safe}_lag1"
        lag2 = f"{safe}_lag2"
        roll3 = f"{safe}_roll3_mean"
        yoy = f"{safe}_yoy_change"

        current = pd.to_numeric(out[base_col], errors="coerce")
        prev1 = pd.to_numeric(out[lag1], errors="coerce") if lag1 in out.columns else pd.Series(np.nan, index=out.index)
        prev2 = pd.to_numeric(out[lag2], errors="coerce") if lag2 in out.columns else pd.Series(np.nan, index=out.index)

        if lag2 in out.columns:
            out[lag2] = prev1.values
        if lag1 in out.columns:
            out[lag1] = current.values
        if roll3 in out.columns:
            out[roll3] = np.nanmean(np.vstack([current.values, prev1.values, prev2.values]), axis=0)
        if yoy in out.columns:
            out[yoy] = current.values - prev1.values

    if "wind_3yr_x_svi" in out.columns and "wind_3yr_sum" in out.columns and "svi_pca_1" in out.columns:
        out["wind_3yr_x_svi"] = pd.to_numeric(out["wind_3yr_sum"], errors="coerce").fillna(0).values * pd.to_numeric(out["svi_pca_1"], errors="coerce").fillna(0).values
    if "seismic_3yr_x_svi" in out.columns and "seismic_3yr_sum" in out.columns and "svi_pca_1" in out.columns:
        out["seismic_3yr_x_svi"] = pd.to_numeric(out["seismic_3yr_sum"], errors="coerce").fillna(0).values * pd.to_numeric(out["svi_pca_1"], errors="coerce").fillna(0).values

    return out

def build_backbone_input(sim_df, forecast_year):
    X = sim_df[feature_cols + cat_cols].copy()
    X["year"] = min(int(forecast_year), int(MODEL_YEAR_CAP))
    return X

def evolve_nonlag_covariates(sim_df, realized_pct, year, scenario):
    out = sim_df.copy()
    realized_pct = np.asarray(realized_pct, dtype=float)

    if "income_growth" in out.columns:
        anchors = out["municipio"].map(muni_income_anchor).fillna(global_income_anchor).astype(float).values
        current = out["income_growth"].astype(float).values
        updated = 0.80 * current + 0.05 * realized_pct + 0.15 * anchors
        out["income_growth"] = np.clip(updated, -6, 6)

    if "establishment_growth" in out.columns:
        anchors = out["municipio"].map(muni_est_anchor).fillna(global_est_anchor).astype(float).values
        current = out["establishment_growth"].astype(float).values
        updated = 0.75 * current + 0.06 * realized_pct + 0.19 * anchors
        out["establishment_growth"] = np.clip(updated, -10, 10)

    if "lag_total_crime_rate" in out.columns:
        anchors = out["municipio"].map(muni_crime_anchor).fillna(global_crime_anchor).astype(float).values
        current = out["lag_total_crime_rate"].astype(float).values
        shock_term = np.where(scenario.get("kind") == "none", 0.0, 0.015 * np.abs(realized_pct))
        updated = 0.85 * current + 0.15 * anchors + shock_term
        out["lag_total_crime_rate"] = np.clip(updated, 0, None)

    return out

def simulate_scenario(latest_rows, scenario_name, scenario, n_sims=N_SIMS, save_detail=SAVE_DETAIL, random_seed=42):
    rng = np.random.default_rng(random_seed)
    base_latest = latest_rows.copy().reset_index(drop=True)

    detail_records = []
    island_records = []
    region_records = []
    muni_records = []

    for sim in range(n_sims):
        sim_df = base_latest.copy()

        for year in FORECAST_YEARS:
            actual_year = int(year)
            sim_df["year"] = actual_year

            annual_overlay = []
            updated_rows = []
            for _, row in sim_df.iterrows():
                row2, adj = apply_scenario_feature_overlay(row, actual_year, scenario)
                annual_overlay.append(adj / 100.0)
                updated_rows.append(row2)
            sim_df = pd.DataFrame(updated_rows)
            sim_df = refresh_recursive_features(sim_df)

            X_curr = build_backbone_input(sim_df, actual_year)
            structural_pct_3y_avg = phase2_backbone_model.predict(X_curr)
            structural_annual = np.array([predicted_annual_pct_to_decimal(v) for v in structural_pct_3y_avg])

            noise = np.array([
                rng.normal(
                    0,
                    max(0.001, min(0.03, 0.40 * residual_std / 100.0 + (muni_vol.get(m, global_vol) / 100.0) * 0.50))
                )
                for m in sim_df["municipio"]
            ])

            annual_rate = structural_annual + np.array(annual_overlay) + noise
            prev_pop = sim_df["total_population"].astype(float).values
            next_pop = np.maximum(prev_pop * (1.0 + annual_rate), 0)

            realized_pct = ((next_pop / np.where(prev_pop == 0, np.nan, prev_pop)) - 1.0) * 100.0
            realized_pct = np.nan_to_num(realized_pct, nan=0.0, posinf=0.0, neginf=0.0)

            sim_df["lag_pop_change_2"] = sim_df["lag_pop_change_1"]
            sim_df["lag_pop_change_1"] = realized_pct
            sim_df["total_population"] = next_pop
            sim_df = evolve_nonlag_covariates(sim_df, realized_pct, actual_year, scenario)
            sim_df = refresh_recursive_features(sim_df, realized_pct=realized_pct)

            if save_detail:
                keep_cols = [c for c in ["municipio","region","year","total_population","lag_pop_change_1","lag_pop_change_2","income_growth","establishment_growth","wind_3yr_sum","seismic_3yr_sum"] if c in sim_df.columns]
                tmp = sim_df[keep_cols].copy()
                tmp["scenario"] = scenario_name
                tmp["sim"] = sim
                detail_records.append(tmp)

            island_total = float(sim_df["total_population"].sum())
            island_records.append({"scenario": scenario_name, "sim": sim, "year": actual_year, "population": island_total})

            reg = sim_df.groupby("region", dropna=False)["total_population"].sum().reset_index()
            reg["scenario"] = scenario_name
            reg["sim"] = sim
            reg["year"] = actual_year
            reg = reg.rename(columns={"total_population":"population"})
            region_records.append(reg)

            muni = sim_df[["municipio","region","year","total_population"]].copy()
            muni["scenario"] = scenario_name
            muni["sim"] = sim
            muni = muni.rename(columns={"total_population":"population"})
            muni_records.append(muni)

    detail_df = pd.concat(detail_records, ignore_index=True) if detail_records else pd.DataFrame()
    island_df = pd.DataFrame(island_records)
    region_df = pd.concat(region_records, ignore_index=True)
    muni_df = pd.concat(muni_records, ignore_index=True)
    return detail_df, island_df, region_df, muni_df

def summarize_simulations(df_sim, group_cols):
    q = (
        df_sim.groupby(group_cols + ["year"])["population"]
        .quantile([0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95])
        .unstack()
        .reset_index()
    )
    q.columns = group_cols + ["year", "p05", "p10", "p25", "p50", "p75", "p90", "p95"]
    return q


Residual std from holdout residuals: 0.3157060256327825


In [15]:

# ----------------------------
# Run scenarios
# ----------------------------
scenario_outputs = {}
timing_records = []

import time
for scen_name, scen in scenario_definitions.items():
    t0 = time.time()
    print("Running:", scen_name)
    detail_df, island_df, region_df, muni_df = simulate_scenario(
        latest_rows=latest_rows,
        scenario_name=scen_name,
        scenario=scen,
        n_sims=N_SIMS,
        save_detail=SAVE_DETAIL,
        random_seed=42
    )
    scenario_outputs[scen_name] = {
        "detail": detail_df,
        "island_sim": island_df,
        "region_sim": region_df,
        "muni_sim": muni_df,
        "island_summary": summarize_simulations(island_df, []),
        "region_summary": summarize_simulations(region_df, ["region"]),
        "muni_summary": summarize_simulations(muni_df, ["municipio","region"]),
    }
    elapsed = time.time() - t0
    timing_records.append({"scenario": scen_name, "seconds": elapsed})
    print(f"Done {scen_name} in {elapsed:.2f}s")

timing_df = pd.DataFrame(timing_records)
saved_files.append(save_csv(timing_df, "phase3_v5_scenario_timing.csv"))
timing_df


Running: baseline
Done baseline in 11.02s
Running: hurricane_cat1
Done hurricane_cat1 in 10.94s
Running: hurricane_cat2
Done hurricane_cat2 in 10.93s
Running: hurricane_cat3
Done hurricane_cat3 in 10.94s
Running: hurricane_cat4
Done hurricane_cat4 in 10.92s
Running: hurricane_cat5_maria_like
Done hurricane_cat5_maria_like in 10.97s
Running: earthquake_m6_0
Done earthquake_m6_0 in 10.96s
Running: earthquake_m6_5
Done earthquake_m6_5 in 11.00s
Running: earthquake_m6_8
Done earthquake_m6_8 in 11.11s
Running: earthquake_m7_2
Done earthquake_m7_2 in 11.38s
Running: earthquake_m7_5
Done earthquake_m7_5 in 11.35s
Saved CSV: phase3_outputs_v5/phase3_v5_scenario_timing.csv


,scenario,seconds
0,baseline,11.020021
1,hurricane_cat1,10.937144
2,hurricane_cat2,10.932730
3,hurricane_cat3,10.939957
4,hurricane_cat4,10.917050
5,hurricane_cat5_maria_like,10.966752
6,earthquake_m6_0,10.956765
7,earthquake_m6_5,11.000755
8,earthquake_m6_8,11.108305
9,earthquake_m7_2,11.378351


In [16]:

# ----------------------------
# Maria backtest (2017 event)
# Uses historical rows up to 2017, then simulates a Maria-like path 2018-2021
# through the final Lasso backbone plus stylized shock overlays.
# ----------------------------
backtest_start_year = 2017
hist_2017 = df[df["year"] == backtest_start_year].copy().sort_values("municipio").reset_index(drop=True)

if hist_2017.empty:
    raise ValueError("Could not build Maria backtest because there are no 2017 rows in the dataset.")

BACKTEST_YEARS = [2018, 2019, 2020, 2021]

def simulate_backtest(latest_rows_2017, scenario, years, n_sims=150, random_seed=42):
    rng = np.random.default_rng(random_seed)
    sim_df0 = latest_rows_2017.copy().reset_index(drop=True)
    island_records = []
    region_records = []

    for sim in range(n_sims):
        sim_df = sim_df0.copy()
        for year in years:
            sim_df["year"] = year

            annual_overlay = []
            updated_rows = []
            for _, row in sim_df.iterrows():
                row2, adj = apply_scenario_feature_overlay(row, year, scenario)
                annual_overlay.append(adj / 100.0)
                updated_rows.append(row2)
            sim_df = pd.DataFrame(updated_rows)
            sim_df = refresh_recursive_features(sim_df)

            X_curr = build_backbone_input(sim_df, year)
            structural_pct_3y_avg = phase2_backbone_model.predict(X_curr)
            structural_annual = np.array([predicted_annual_pct_to_decimal(v) for v in structural_pct_3y_avg])

            noise = np.array([
                rng.normal(
                    0,
                    max(0.001, min(0.03, 0.40 * residual_std / 100.0 + (muni_vol.get(m, global_vol) / 100.0) * 0.50))
                )
                for m in sim_df["municipio"]
            ])

            annual_rate = structural_annual + np.array(annual_overlay) + noise
            prev_pop = sim_df["total_population"].astype(float).values
            next_pop = np.maximum(prev_pop * (1.0 + annual_rate), 0)
            realized_pct = ((next_pop / np.where(prev_pop == 0, np.nan, prev_pop)) - 1.0) * 100.0
            realized_pct = np.nan_to_num(realized_pct, nan=0.0, posinf=0.0, neginf=0.0)

            sim_df["lag_pop_change_2"] = sim_df["lag_pop_change_1"]
            sim_df["lag_pop_change_1"] = realized_pct
            sim_df["total_population"] = next_pop
            sim_df = evolve_nonlag_covariates(sim_df, realized_pct, year, scenario)
            sim_df = refresh_recursive_features(sim_df, realized_pct=realized_pct)

            island_records.append({
                "sim": sim, "year": year, "population": float(sim_df["total_population"].sum())
            })

            reg = sim_df.groupby("region", dropna=False)["total_population"].sum().reset_index()
            reg["sim"] = sim
            reg["year"] = year
            reg = reg.rename(columns={"total_population": "population"})
            region_records.append(reg)

    island_df = pd.DataFrame(island_records)
    region_df = pd.concat(region_records, ignore_index=True)
    return island_df, region_df

maria_backtest_scenario = compute_maria_backtest_shock(2017)
maria_bt_island_sim, maria_bt_region_sim = simulate_backtest(hist_2017, maria_backtest_scenario, BACKTEST_YEARS)

maria_bt_island = summarize_simulations(maria_bt_island_sim, [])
maria_bt_region = summarize_simulations(maria_bt_region_sim, ["region"])

obs_island = df.groupby("year", as_index=False)["total_population"].sum().rename(columns={"total_population": "observed_population"})
obs_region = df.groupby(["region", "year"], as_index=False)["total_population"].sum().rename(columns={"total_population": "observed_population"})

maria_bt_island = maria_bt_island.merge(obs_island, on="year", how="left")
maria_bt_region = maria_bt_region.merge(obs_region, on=["region", "year"], how="left")

maria_bt_island["abs_error"] = maria_bt_island["p50"] - maria_bt_island["observed_population"]
maria_bt_island["pct_error"] = np.where(
    maria_bt_island["observed_population"].abs() > 0,
    100 * maria_bt_island["abs_error"] / maria_bt_island["observed_population"],
    np.nan
)

maria_bt_region["abs_error"] = maria_bt_region["p50"] - maria_bt_region["observed_population"]
maria_bt_region["pct_error"] = np.where(
    maria_bt_region["observed_population"].abs() > 0,
    100 * maria_bt_region["abs_error"] / maria_bt_region["observed_population"],
    np.nan
)

saved_files.append(save_csv(maria_bt_island, "phase3_v5_maria_backtest_island.csv"))
saved_files.append(save_csv(maria_bt_region, "phase3_v5_maria_backtest_region.csv"))

maria_bt_island.head()


Saved CSV: phase3_outputs_v5/phase3_v5_maria_backtest_island.csv
Saved CSV: phase3_outputs_v5/phase3_v5_maria_backtest_region.csv


,year,p05,p10,p25,p50,p75,p90,p95,observed_population,abs_error,pct_error
0,2018,3.320703e+06,3.322982e+06,3.327875e+06,3.331447e+06,3.335721e+06,3.340329e+06,3.341620e+06,3386941.0,-55494.263226,-1.638477
1,2019,3.156856e+06,3.161051e+06,3.170059e+06,3.179653e+06,3.188559e+06,3.195497e+06,3.199887e+06,3318447.0,-138793.566886,-4.182486
2,2020,3.013804e+06,3.017957e+06,3.030318e+06,3.045090e+06,3.058774e+06,3.071587e+06,3.078296e+06,3255642.0,-210552.347756,-6.467307
3,2021,2.886787e+06,2.891323e+06,2.903227e+06,2.922757e+06,2.940762e+06,2.960612e+06,2.970135e+06,3311274.0,-388517.271660,-11.733166


In [17]:

# ----------------------------
# Combine scenario summaries
# ----------------------------
island_summary_all = []
region_summary_all = []
muni_summary_all = []

for scen_name, out in scenario_outputs.items():
    a = out["island_summary"].copy()
    a["scenario"] = scen_name
    island_summary_all.append(a)

    b = out["region_summary"].copy()
    b["scenario"] = scen_name
    region_summary_all.append(b)

    c = out["muni_summary"].copy()
    c["scenario"] = scen_name
    muni_summary_all.append(c)

island_summary_all = pd.concat(island_summary_all, ignore_index=True)
region_summary_all = pd.concat(region_summary_all, ignore_index=True)
muni_summary_all = pd.concat(muni_summary_all, ignore_index=True)

saved_files.append(save_csv(island_summary_all, "phase3_v5_island_scenario_summary.csv"))
saved_files.append(save_csv(region_summary_all, "phase3_v5_region_scenario_summary.csv"))
saved_files.append(save_csv(muni_summary_all, "phase3_v5_municipal_scenario_summary.csv"))

island_summary_all.head()


Saved CSV: phase3_outputs_v5/phase3_v5_island_scenario_summary.csv
Saved CSV: phase3_outputs_v5/phase3_v5_region_scenario_summary.csv
Saved CSV: phase3_outputs_v5/phase3_v5_municipal_scenario_summary.csv


,year,p05,p10,p25,p50,p75,p90,p95,scenario
0,2025,3.161985e+06,3.164354e+06,3.168124e+06,3.171816e+06,3.176765e+06,3.180430e+06,3.182228e+06,baseline
1,2026,3.082798e+06,3.087169e+06,3.094093e+06,3.102981e+06,3.112038e+06,3.120430e+06,3.123862e+06,baseline
2,2027,3.000851e+06,3.004099e+06,3.014621e+06,3.031603e+06,3.043142e+06,3.055902e+06,3.063533e+06,baseline
3,2028,2.913944e+06,2.920000e+06,2.935163e+06,2.954742e+06,2.972793e+06,2.991581e+06,3.000008e+06,baseline
4,2029,2.826654e+06,2.837659e+06,2.855134e+06,2.878385e+06,2.899893e+06,2.923592e+06,2.933825e+06,baseline


In [18]:

# ----------------------------
# Impact summaries relative to baseline
# ----------------------------
baseline_island = island_summary_all[island_summary_all["scenario"] == "baseline"][["year","p50"]].rename(columns={"p50":"baseline_p50"})
baseline_region = region_summary_all[region_summary_all["scenario"] == "baseline"][["region","year","p50"]].rename(columns={"p50":"baseline_p50"})
baseline_muni = muni_summary_all[muni_summary_all["scenario"] == "baseline"][["municipio","region","year","p50"]].rename(columns={"p50":"baseline_p50"})

island_impact = island_summary_all.merge(baseline_island, on="year", how="left")
island_impact["abs_impact_vs_baseline"] = island_impact["p50"] - island_impact["baseline_p50"]
island_impact["pct_impact_vs_baseline"] = 100 * island_impact["abs_impact_vs_baseline"] / island_impact["baseline_p50"]

region_impact = region_summary_all.merge(baseline_region, on=["region","year"], how="left")
region_impact["abs_impact_vs_baseline"] = region_impact["p50"] - region_impact["baseline_p50"]
region_impact["pct_impact_vs_baseline"] = 100 * region_impact["abs_impact_vs_baseline"] / region_impact["baseline_p50"]

muni_impact = muni_summary_all.merge(baseline_muni, on=["municipio","region","year"], how="left")
muni_impact["abs_impact_vs_baseline"] = muni_impact["p50"] - muni_impact["baseline_p50"]
muni_impact["pct_impact_vs_baseline"] = 100 * muni_impact["abs_impact_vs_baseline"] / muni_impact["baseline_p50"]

saved_files.append(save_csv(island_impact, "phase3_v5_island_impact_summary.csv"))
saved_files.append(save_csv(region_impact, "phase3_v5_region_impact_summary.csv"))

region_2030 = region_impact[region_impact["year"] == 2030].copy()
muni_2030 = muni_impact[muni_impact["year"] == 2030].copy()

saved_files.append(save_csv(region_2030, "phase3_v5_region_2030_comparison.csv"))
saved_files.append(save_csv(muni_2030, "phase3_v5_municipal_2030_comparison.csv"))

island_impact.head()


Saved CSV: phase3_outputs_v5/phase3_v5_island_impact_summary.csv
Saved CSV: phase3_outputs_v5/phase3_v5_region_impact_summary.csv
Saved CSV: phase3_outputs_v5/phase3_v5_region_2030_comparison.csv
Saved CSV: phase3_outputs_v5/phase3_v5_municipal_2030_comparison.csv


,year,p05,p10,p25,p50,p75,p90,p95,scenario,baseline_p50,abs_impact_vs_baseline,pct_impact_vs_baseline
0,2025,3.161985e+06,3.164354e+06,3.168124e+06,3.171816e+06,3.176765e+06,3.180430e+06,3.182228e+06,baseline,3.171816e+06,0.0,0.0
1,2026,3.082798e+06,3.087169e+06,3.094093e+06,3.102981e+06,3.112038e+06,3.120430e+06,3.123862e+06,baseline,3.102981e+06,0.0,0.0
2,2027,3.000851e+06,3.004099e+06,3.014621e+06,3.031603e+06,3.043142e+06,3.055902e+06,3.063533e+06,baseline,3.031603e+06,0.0,0.0
3,2028,2.913944e+06,2.920000e+06,2.935163e+06,2.954742e+06,2.972793e+06,2.991581e+06,3.000008e+06,baseline,2.954742e+06,0.0,0.0
4,2029,2.826654e+06,2.837659e+06,2.855134e+06,2.878385e+06,2.899893e+06,2.923592e+06,2.933825e+06,baseline,2.878385e+06,0.0,0.0


In [19]:
# ----------------------------
# Scenario severity ranking tables (2030 impact vs baseline)
# ----------------------------
island_2030_rank = (
    island_impact[island_impact["year"] == 2030]
    .loc[lambda d: d["scenario"] != "baseline", ["scenario", "p50", "baseline_p50", "abs_impact_vs_baseline", "pct_impact_vs_baseline"]]
    .sort_values("abs_impact_vs_baseline")
    .reset_index(drop=True)
)

region_2030_rank = (
    region_impact[region_impact["year"] == 2030]
    .loc[lambda d: d["scenario"] != "baseline", ["scenario", "region", "p50", "baseline_p50", "abs_impact_vs_baseline", "pct_impact_vs_baseline"]]
    .sort_values(["scenario", "abs_impact_vs_baseline"])
    .reset_index(drop=True)
)

muni_2030_rank = (
    muni_impact[muni_impact["year"] == 2030]
    .loc[lambda d: d["scenario"] != "baseline", ["scenario", "municipio", "region", "p50", "baseline_p50", "abs_impact_vs_baseline", "pct_impact_vs_baseline"]]
    .sort_values(["scenario", "abs_impact_vs_baseline"])
    .reset_index(drop=True)
)

saved_files.append(save_csv(island_2030_rank, "phase3_v5_island_2030_severity_rank.csv"))
saved_files.append(save_csv(region_2030_rank, "phase3_v5_region_2030_severity_rank.csv"))
saved_files.append(save_csv(muni_2030_rank, "phase3_v5_municipio_2030_severity_rank.csv"))

print("Island 2030 severity ranking:")
display(island_2030_rank.head(10))

Saved CSV: phase3_outputs_v5/phase3_v5_island_2030_severity_rank.csv
Saved CSV: phase3_outputs_v5/phase3_v5_region_2030_severity_rank.csv
Saved CSV: phase3_outputs_v5/phase3_v5_municipio_2030_severity_rank.csv
Island 2030 severity ranking:


,scenario,p50,baseline_p50,abs_impact_vs_baseline,pct_impact_vs_baseline
0,hurricane_cat5_maria_like,2.335409e+06,2.800503e+06,-465094.338661,-16.607528
1,hurricane_cat4,2.477597e+06,2.800503e+06,-322905.773511,-11.530277
2,hurricane_cat3,2.592899e+06,2.800503e+06,-207604.013244,-7.413097
3,earthquake_m7_2,2.598869e+06,2.800503e+06,-201633.694285,-7.199910
4,earthquake_m7_5,2.598869e+06,2.800503e+06,-201633.694285,-7.199910
5,earthquake_m6_8,2.642648e+06,2.800503e+06,-157855.064907,-5.636668
6,hurricane_cat2,2.677851e+06,2.800503e+06,-122652.436454,-4.379657
7,earthquake_m6_5,2.687019e+06,2.800503e+06,-113484.533086,-4.052291
8,earthquake_m6_0,2.731987e+06,2.800503e+06,-68516.074366,-2.446563
9,hurricane_cat1,2.738628e+06,2.800503e+06,-61874.667474,-2.209413


## Scenario-summary note

The impact tables below rank scenarios by **2030 median population effect relative to baseline** at the island, region, and municipality levels.  
These tables are useful for slides because they summarize the scenario engine in one place and make it easier to compare the severity ordering across hazards.

In [20]:

# ----------------------------
# Plot helpers
# ----------------------------
observed_island = (
    df.groupby("year", as_index=False)["total_population"].sum()
      .rename(columns={"total_population":"population"})
)

def add_chart_record(filename, title, description):
    chart_index_records.append({
        "filename": filename,
        "title": title,
        "description": description
    })

def plot_fan(ax, summary_df, label_prefix="", median_color=None, fill_alpha=0.15, add_label=True):
    ax.fill_between(summary_df["year"], summary_df["p05"], summary_df["p95"], alpha=fill_alpha)
    ax.fill_between(summary_df["year"], summary_df["p10"], summary_df["p90"], alpha=fill_alpha + 0.05)
    ax.fill_between(summary_df["year"], summary_df["p25"], summary_df["p75"], alpha=fill_alpha + 0.08)
    ax.plot(summary_df["year"], summary_df["p50"], linewidth=2)

def style_observed(ax):
    ax.plot(observed_island["year"], observed_island["population"], marker="o", linewidth=2, label="Observed")
    ax.axvline(latest_year, linestyle="--", linewidth=1.5)
    ax.set_xlabel("Year")
    ax.set_ylabel("Population")
    ax.grid(True, alpha=0.3)


In [21]:

# ----------------------------
# Chart 1: all scenario island fan charts
# ----------------------------
scens = ["baseline","hurricane_cat3","hurricane_cat5_maria_like","earthquake_m6_5","earthquake_m6_8","earthquake_m7_5"]
scens = [s for s in scens if s in island_summary_all["scenario"].unique()]

n = len(scens)
fig, axes = plt.subplots(2, math.ceil(n/2), figsize=(18, 8), sharey=True)
axes = np.array(axes).reshape(-1)

for ax, scen in zip(axes, scens):
    style_observed(ax)
    s = island_summary_all[island_summary_all["scenario"] == scen].sort_values("year")
    plot_fan(ax, s)
    ax.set_title(scen.replace("_"," ").title())

for ax in axes[n:]:
    ax.axis("off")

fig.suptitle("Island-level fan charts by scenario", fontsize=16)
fname = "phase3_v5_island_fan_charts_all_scenarios.png"
saved_files.append(save_fig(fig, fname))
add_chart_record(fname, "Island fan charts by scenario", "Observed island history with forecast fan charts for baseline and selected hazard scenarios.")


Saved FIG: phase3_outputs_v5/phase3_v5_island_fan_charts_all_scenarios.png


In [22]:

# ----------------------------
# Chart 2: baseline vs hurricane ladder
# ----------------------------
h_ladder = [s for s in ["baseline","hurricane_cat1","hurricane_cat2","hurricane_cat3","hurricane_cat4","hurricane_cat5_maria_like"] if s in island_summary_all["scenario"].unique()]
fig, ax = plt.subplots(figsize=(12, 6))
style_observed(ax)
for scen in h_ladder:
    s = island_summary_all[island_summary_all["scenario"] == scen].sort_values("year")
    ax.plot(s["year"], s["p50"], linewidth=2, label=scen.replace("_"," ").title())
ax.legend()
ax.set_title("Island median forecast paths — hurricane magnitude ladder")
fname = "phase3_v5_island_hurricane_ladder.png"
saved_files.append(save_fig(fig, fname))
add_chart_record(fname, "Island hurricane ladder", "Median island forecast paths comparing baseline against Cat 1 through Cat 5 Maria-like hurricane scenarios.")


Saved FIG: phase3_outputs_v5/phase3_v5_island_hurricane_ladder.png


In [23]:

# ----------------------------
# Chart 3: baseline vs earthquake ladder
# ----------------------------
e_ladder = [s for s in ["baseline","earthquake_m6_0","earthquake_m6_5","earthquake_m6_8","earthquake_m7_2","earthquake_m7_5"] if s in island_summary_all["scenario"].unique()]
fig, ax = plt.subplots(figsize=(12, 6))
style_observed(ax)
for scen in e_ladder:
    s = island_summary_all[island_summary_all["scenario"] == scen].sort_values("year")
    ax.plot(s["year"], s["p50"], linewidth=2, label=scen.replace("_"," ").title())
ax.legend()
ax.set_title("Island median forecast paths — earthquake magnitude ladder")
fname = "phase3_v5_island_earthquake_ladder.png"
saved_files.append(save_fig(fig, fname))
add_chart_record(fname, "Island earthquake ladder", "Median island forecast paths comparing baseline against multiple earthquake magnitude scenarios.")


Saved FIG: phase3_outputs_v5/phase3_v5_island_earthquake_ladder.png


In [24]:

# ----------------------------
# Chart 4: side-by-side baseline vs selected hazards
# ----------------------------
pairs = [("hurricane_cat3", "No disaster vs Hurricane Cat 3"),
         ("hurricane_cat5_maria_like", "No disaster vs Cat 5 Maria-like"),
         ("earthquake_m6_8", "No disaster vs Earthquake M6.8"),
         ("earthquake_m7_5", "No disaster vs Earthquake M7.5")]

fig, axes = plt.subplots(2, 2, figsize=(16, 10), sharey=True)
axes = axes.flatten()
base = island_summary_all[island_summary_all["scenario"] == "baseline"].sort_values("year")

for ax, (scen, title) in zip(axes, pairs):
    style_observed(ax)
    plot_fan(ax, base)
    if scen in island_summary_all["scenario"].unique():
        s = island_summary_all[island_summary_all["scenario"] == scen].sort_values("year")
        ax.plot(base["year"], base["p50"], linestyle="--", linewidth=2, label="Baseline median")
        ax.plot(s["year"], s["p50"], linewidth=2.5, label="Scenario median")
    ax.set_title(title)
    ax.legend()

fig.suptitle("Baseline vs selected hazard scenarios", fontsize=16)
fname = "phase3_v5_side_by_side_baseline_vs_hazards.png"
saved_files.append(save_fig(fig, fname))
add_chart_record(fname, "Baseline vs selected hazards", "Side-by-side island-level comparison panels for baseline against selected hurricane and earthquake scenarios.")


Saved FIG: phase3_outputs_v5/phase3_v5_side_by_side_baseline_vs_hazards.png


In [25]:

# ----------------------------
# Chart 5: island impact over time relative to baseline
# ----------------------------
fig, ax = plt.subplots(figsize=(12, 6))
for scen in [s for s in island_impact["scenario"].unique() if s != "baseline"]:
    s = island_impact[island_impact["scenario"] == scen].sort_values("year")
    ax.plot(s["year"], s["abs_impact_vs_baseline"], linewidth=2, label=scen.replace("_"," ").title())
ax.axhline(0, color="black", linewidth=1)
ax.set_title("Island median impact vs baseline over time")
ax.set_xlabel("Year")
ax.set_ylabel("Population difference vs baseline")
ax.grid(True, alpha=0.3)
ax.legend(ncol=2)
fname = "phase3_v5_island_impact_over_time.png"
saved_files.append(save_fig(fig, fname))
add_chart_record(fname, "Island impact over time", "Difference between scenario median and baseline median island population by year.")


Saved FIG: phase3_outputs_v5/phase3_v5_island_impact_over_time.png


In [26]:

# ----------------------------
# Chart 6: Maria backtest island
# ----------------------------
fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(obs_island["year"], obs_island["observed_population"], marker="o", linewidth=2, label="Observed island population")
ax.plot(maria_bt_island["year"], maria_bt_island["p50"], marker="o", linewidth=2, label="Maria-like model path")
ax.axvline(2017, color="red", linestyle="--", linewidth=1.5, label="Maria year")
ax.set_title("Maria backtest: observed vs Maria-like modeled island path")
ax.set_xlabel("Year")
ax.set_ylabel("Population")
ax.grid(True, alpha=0.3)
ax.legend()
fname = "phase3_v5_maria_backtest_island.png"
saved_files.append(save_fig(fig, fname))
add_chart_record(fname, "Maria backtest island", "Observed island population compared with Maria-like modeled backtest path from 2018-2021.")


Saved FIG: phase3_outputs_v5/phase3_v5_maria_backtest_island.png


In [27]:

# ----------------------------
# Chart 7: region 2030 impact bar charts for selected scenarios
# ----------------------------
selected_scenarios = [s for s in ["hurricane_cat3","hurricane_cat5_maria_like","earthquake_m6_8","earthquake_m7_5"] if s in region_2030["scenario"].unique()]
fig, axes = plt.subplots(2, 2, figsize=(16, 10), sharey=True)
axes = axes.flatten()

for ax, scen in zip(axes, selected_scenarios):
    s = region_2030[region_2030["scenario"] == scen].sort_values("abs_impact_vs_baseline")
    ax.barh(s["region"], s["abs_impact_vs_baseline"])
    ax.set_title(f"2030 region impact vs baseline — {scen.replace('_',' ').title()}")
    ax.set_xlabel("Population difference vs baseline")

for ax in axes[len(selected_scenarios):]:
    ax.axis("off")

fname = "phase3_v5_region_2030_impact_bars.png"
saved_files.append(save_fig(fig, fname))
add_chart_record(fname, "Region 2030 impact bars", "Horizontal bar charts showing region-level 2030 population impact versus baseline for selected scenarios.")


Saved FIG: phase3_outputs_v5/phase3_v5_region_2030_impact_bars.png


In [28]:

# ----------------------------
# Chart 8: regional faceted median paths for Maria-like and Cat 3
# ----------------------------
faceted_scenarios = [s for s in ["baseline","hurricane_cat3","hurricane_cat5_maria_like"] if s in region_summary_all["scenario"].unique()]
regions = [r for r in sorted(region_summary_all["region"].dropna().unique()) if r != "Unknown"]
n = len(regions)
fig, axes = plt.subplots(math.ceil(n/3), 3, figsize=(18, 4*math.ceil(n/3)), sharex=True, sharey=False)
axes = np.array(axes).reshape(-1)

for ax, region in zip(axes, regions):
    obs_r = df[df["region"] == region].groupby("year", as_index=False)["total_population"].sum()
    ax.plot(obs_r["year"], obs_r["total_population"], marker="o", linewidth=2, label="Observed")
    ax.axvline(latest_year, linestyle="--", linewidth=1.2)
    for scen in faceted_scenarios:
        s = region_summary_all[(region_summary_all["region"] == region) & (region_summary_all["scenario"] == scen)].sort_values("year")
        ax.plot(s["year"], s["p50"], linewidth=2, label=scen.replace("_"," ").title())
    ax.set_title(region)
    ax.grid(True, alpha=0.3)

for ax in axes[n:]:
    ax.axis("off")

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", ncol=4)
fig.suptitle("Regional median paths — baseline, Cat 3, and Cat 5 Maria-like", fontsize=16, y=0.98)
fname = "phase3_v5_region_faceted_paths.png"
saved_files.append(save_fig(fig, fname))
add_chart_record(fname, "Regional faceted paths", "Faceted regional median path charts comparing baseline, hurricane Cat 3, and Cat 5 Maria-like scenarios.")


Saved FIG: phase3_outputs_v5/phase3_v5_region_faceted_paths.png


In [29]:

# ----------------------------
# Chart 9: municipal 2030 top losses under Maria-like
# ----------------------------
if "hurricane_cat5_maria_like" in muni_2030["scenario"].unique():
    s = muni_2030[muni_2030["scenario"] == "hurricane_cat5_maria_like"].sort_values("abs_impact_vs_baseline").head(15)
    fig, ax = plt.subplots(figsize=(12, 7))
    ax.barh(s["municipio"], s["abs_impact_vs_baseline"])
    ax.set_title("Top municipal 2030 losses vs baseline — Cat 5 Maria-like")
    ax.set_xlabel("Population difference vs baseline")
    fname = "phase3_v5_top_municipal_losses_maria_like.png"
    saved_files.append(save_fig(fig, fname))
    add_chart_record(fname, "Top municipal losses Maria-like", "Municipalities with the largest modeled 2030 population losses relative to baseline under a Cat 5 Maria-like hurricane.")


Saved FIG: phase3_outputs_v5/phase3_v5_top_municipal_losses_maria_like.png


In [30]:

# ----------------------------
# Chart 10: region heatmap for cumulative impact (2026-2030)
# ----------------------------
heat = (
    region_impact[region_impact["scenario"].isin([s for s in selected_scenarios])]
    .groupby(["scenario","region","year"], as_index=False)["abs_impact_vs_baseline"].mean()
)
for scen in heat["scenario"].unique():
    pivot = heat[heat["scenario"] == scen].pivot(index="region", columns="year", values="abs_impact_vs_baseline")
    fig, ax = plt.subplots(figsize=(10, 5))
    im = ax.imshow(pivot.values, aspect="auto")
    ax.set_xticks(range(len(pivot.columns)))
    ax.set_xticklabels(pivot.columns)
    ax.set_yticks(range(len(pivot.index)))
    ax.set_yticklabels(pivot.index)
    ax.set_title(f"Region impact heatmap — {scen.replace('_',' ').title()}")
    ax.set_xlabel("Year")
    ax.set_ylabel("Region")
    fig.colorbar(im, ax=ax, label="Population difference vs baseline")
    fname = f"phase3_v5_heatmap_{scen}.png"
    saved_files.append(save_fig(fig, fname))
    add_chart_record(fname, f"Heatmap {scen}", f"Regional impact heatmap over forecast years for {scen.replace('_',' ')}.")


Saved FIG: phase3_outputs_v5/phase3_v5_heatmap_earthquake_m6_8.png
Saved FIG: phase3_outputs_v5/phase3_v5_heatmap_earthquake_m7_5.png
Saved FIG: phase3_outputs_v5/phase3_v5_heatmap_hurricane_cat3.png
Saved FIG: phase3_outputs_v5/phase3_v5_heatmap_hurricane_cat5_maria_like.png


In [31]:

# ----------------------------
# Chart 11: uncertainty width comparison
# ----------------------------
unc = island_summary_all.copy()
unc["width_90"] = unc["p95"] - unc["p05"]
fig, ax = plt.subplots(figsize=(12, 6))
for scen in [s for s in island_summary_all["scenario"].unique()]:
    s = unc[unc["scenario"] == scen].sort_values("year")
    ax.plot(s["year"], s["width_90"], linewidth=2, label=scen.replace("_"," ").title())
ax.set_title("Island uncertainty width over time (95%-5%)")
ax.set_xlabel("Year")
ax.set_ylabel("Width of uncertainty band")
ax.grid(True, alpha=0.3)
ax.legend(ncol=2)
fname = "phase3_v5_uncertainty_width_comparison.png"
saved_files.append(save_fig(fig, fname))
add_chart_record(fname, "Uncertainty width comparison", "Comparison of island forecast uncertainty band width over time across scenarios.")


Saved FIG: phase3_outputs_v5/phase3_v5_uncertainty_width_comparison.png


## Additional presentation-ready charts and diagnostics

These charts improve the notebook for slide generation by showing:
- modeled vs observed Maria backtest error,
- percent impact relative to baseline,
- a region-by-scenario 2030 heatmap,
- vulnerability vs modeled impact at the municipal level,
- and direct comparison of top municipal losses across major hazards.

These are meant to complement the existing island, region, and municipal scenario figures.


In [32]:

# ----------------------------
# Chart 12: Maria backtest error / deviation from observed
# ----------------------------
# maria_bt_island already contains observed_population from the earlier backtest merge.
# Re-merge only if that column is not present. If a re-merge creates suffixes, normalize them.
bt_compare = maria_bt_island.copy()

if "observed_population" not in bt_compare.columns:
    bt_compare = bt_compare.merge(
        obs_island[["year", "observed_population"]],
        on="year",
        how="left",
        suffixes=("", "_obs")
    ).copy()

# Normalize any suffixed observed-population columns that may have appeared after a merge.
if "observed_population" not in bt_compare.columns:
    for cand in ["observed_population_obs", "observed_population_x", "observed_population_y"]:
        if cand in bt_compare.columns:
            bt_compare["observed_population"] = bt_compare[cand]
            break

if "observed_population" not in bt_compare.columns:
    raise KeyError(
        "Could not find observed_population in Maria backtest data. "
        f"Available columns: {list(bt_compare.columns)}"
    )

bt_compare = bt_compare[bt_compare["year"].between(2018, 2021)].copy()
bt_compare["model_minus_observed"] = bt_compare["p50"] - bt_compare["observed_population"]
bt_compare["pct_diff_vs_observed"] = np.where(
    bt_compare["observed_population"].abs() > 0,
    100 * bt_compare["model_minus_observed"] / bt_compare["observed_population"],
    np.nan
)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
axes[0].bar(bt_compare["year"].astype(str), bt_compare["model_minus_observed"])
axes[0].axhline(0, color="black", linewidth=1)
axes[0].set_title("Maria backtest error by year")
axes[0].set_ylabel("Modeled - observed population")

axes[1].bar(bt_compare["year"].astype(str), bt_compare["pct_diff_vs_observed"])
axes[1].axhline(0, color="black", linewidth=1)
axes[1].set_title("Maria backtest percent error by year")
axes[1].set_ylabel("% difference vs observed")

fname = "phase3_v5_maria_backtest_error.png"
saved_files.append(save_fig(fig, fname))
add_chart_record(
    fname,
    "Maria backtest error",
    "Two-panel chart showing the Maria-like island backtest deviation from observed population in levels and percent terms."
)


Saved FIG: phase3_outputs_v5/phase3_v5_maria_backtest_error.png


In [33]:

# ----------------------------
# Chart 13: island percent impact relative to baseline
# ----------------------------
impact_pct = island_impact.copy()
impact_pct["pct_impact_vs_baseline"] = np.where(
    impact_pct["baseline_p50"].abs() > 0,
    100 * impact_pct["abs_impact_vs_baseline"] / impact_pct["baseline_p50"],
    np.nan
)

fig, ax = plt.subplots(figsize=(12, 6))
focus_scenarios = [s for s in ["hurricane_cat3", "hurricane_cat5_maria_like", "earthquake_m6_8", "earthquake_m7_5"] if s in impact_pct["scenario"].unique()]
for scen in focus_scenarios:
    s = impact_pct[impact_pct["scenario"] == scen].sort_values("year")
    ax.plot(s["year"], s["pct_impact_vs_baseline"], linewidth=2, label=scen.replace("_", " ").title())
ax.axhline(0, color="black", linewidth=1)
ax.set_title("Island impact relative to baseline over time")
ax.set_xlabel("Year")
ax.set_ylabel("% difference vs baseline population")
ax.grid(True, alpha=0.3)
ax.legend()

fname = "phase3_v5_island_pct_impact_over_time.png"
saved_files.append(save_fig(fig, fname))
add_chart_record(fname, "Island percent impact over time", "Percent difference between scenario median and baseline median island population over forecast years.")


Saved FIG: phase3_outputs_v5/phase3_v5_island_pct_impact_over_time.png


In [34]:

# ----------------------------
# Chart 14: region x scenario heatmap for 2030 impact
# ----------------------------
heat2030 = region_2030.copy()
if not heat2030.empty:
    scen_order = [s for s in ["hurricane_cat1","hurricane_cat2","hurricane_cat3","hurricane_cat4","hurricane_cat5_maria_like","earthquake_m6_0","earthquake_m6_5","earthquake_m6_8","earthquake_m7_2","earthquake_m7_5"] if s in heat2030["scenario"].unique()]
    pivot = heat2030.pivot(index="region", columns="scenario", values="abs_impact_vs_baseline")
    pivot = pivot[[c for c in scen_order if c in pivot.columns]] if scen_order else pivot
    fig, ax = plt.subplots(figsize=(14, 6))
    im = ax.imshow(pivot.values, aspect="auto")
    ax.set_xticks(range(len(pivot.columns)))
    ax.set_xticklabels([c.replace("_", " ").title() for c in pivot.columns], rotation=45, ha="right")
    ax.set_yticks(range(len(pivot.index)))
    ax.set_yticklabels(pivot.index)
    ax.set_title("Region-by-scenario 2030 impact heatmap")
    ax.set_xlabel("Scenario")
    ax.set_ylabel("Region")
    cbar = fig.colorbar(im, ax=ax)
    cbar.set_label("Population difference vs baseline")
    fname = "phase3_v5_region_scenario_heatmap_2030.png"
    saved_files.append(save_fig(fig, fname))
    add_chart_record(fname, "Region-scenario heatmap 2030", "Heatmap of region-level 2030 population impact versus baseline across hurricane and earthquake magnitude scenarios.")


Saved FIG: phase3_outputs_v5/phase3_v5_region_scenario_heatmap_2030.png


In [35]:

# ----------------------------
# Chart 15: municipal vulnerability vs impact scatter (Maria-like 2030)
# ----------------------------
if "hurricane_cat5_maria_like" in muni_2030["scenario"].unique():
    latest_baseline = (
        df.sort_values(["municipio", "year"])
          .groupby("municipio", as_index=False)
          .tail(1)[["municipio", "region", "svi_pca_1", "baseline_vulnerability_index", "total_population"]]
          .drop_duplicates("municipio")
    )
    scat = muni_2030[muni_2030["scenario"] == "hurricane_cat5_maria_like"].merge(
        latest_baseline, on="municipio", how="left", suffixes=("", "_base")
    ).copy()

    # Normalize possible region column naming after merge
    if "region" not in scat.columns:
        for cand in ["region_base", "region_x", "region_y"]:
            if cand in scat.columns:
                scat["region"] = scat[cand]
                break

    # Last-resort region mapping from the master dataframe
    if "region" not in scat.columns:
        region_lookup = (
            df[["municipio", "region"]]
            .dropna()
            .drop_duplicates("municipio")
        )
        scat = scat.merge(region_lookup, on="municipio", how="left", suffixes=("", "_lookup"))
        if "region" not in scat.columns and "region_lookup" in scat.columns:
            scat["region"] = scat["region_lookup"]

    # Pick vulnerability column robustly
    if "svi_pca_1" in scat.columns:
        xcol = "svi_pca_1"
    elif "baseline_vulnerability_index" in scat.columns:
        xcol = "baseline_vulnerability_index"
    else:
        raise KeyError("Could not find a vulnerability column for the municipal scatter plot.")

    fig, ax = plt.subplots(figsize=(11, 7))
    group_col = "region" if "region" in scat.columns else None

    if group_col is not None:
        for region_name, grp in scat.groupby(group_col, dropna=False):
            label = region_name if pd.notna(region_name) else "Unknown region"
            ax.scatter(grp[xcol], grp["abs_impact_vs_baseline"], label=label, alpha=0.75)
        ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
    else:
        ax.scatter(scat[xcol], scat["abs_impact_vs_baseline"], alpha=0.75)

    ax.axhline(0, color="black", linewidth=1)
    ax.set_title("Municipal vulnerability vs modeled 2030 impact — Cat 5 Maria-like")
    ax.set_xlabel("Vulnerability metric")
    ax.set_ylabel("Population difference vs baseline")
    ax.grid(True, alpha=0.3)
    fname = "phase3_v5_municipal_vulnerability_vs_impact_scatter.png"
    saved_files.append(save_fig(fig, fname))
    add_chart_record(fname, "Municipal vulnerability vs impact", "Scatter plot showing whether more vulnerable municipalities experience larger modeled 2030 losses under the Cat 5 Maria-like scenario.")


Saved FIG: phase3_outputs_v5/phase3_v5_municipal_vulnerability_vs_impact_scatter.png


In [36]:

# ----------------------------
# Chart 16: top municipal losses comparison (Maria-like vs M7.5)
# ----------------------------
compare_scenarios = [s for s in ["hurricane_cat5_maria_like", "earthquake_m7_5"] if s in muni_2030["scenario"].unique()]
if compare_scenarios:
    top_rows = []
    for scen in compare_scenarios:
        tmp = (
            muni_2030[muni_2030["scenario"] == scen]
            .sort_values("abs_impact_vs_baseline")
            .head(10)
            .copy()
        )
        tmp["scenario_label"] = scen.replace("_", " ").title()
        top_rows.append(tmp)
    top_compare = pd.concat(top_rows, ignore_index=True)
    fig, axes = plt.subplots(1, len(compare_scenarios), figsize=(7*len(compare_scenarios), 7), sharex=False)
    if len(compare_scenarios) == 1:
        axes = [axes]
    for ax, scen in zip(axes, compare_scenarios):
        s = top_compare[top_compare["scenario_label"] == scen.replace("_", " ").title()].sort_values("abs_impact_vs_baseline")
        ax.barh(s["municipio"], s["abs_impact_vs_baseline"])
        ax.set_title(f"Top municipal losses — {scen.replace('_', ' ').title()}")
        ax.set_xlabel("Population difference vs baseline")
    fname = "phase3_v5_top_municipal_losses_compare.png"
    saved_files.append(save_fig(fig, fname))
    add_chart_record(fname, "Top municipal losses comparison", "Side-by-side comparison of the municipalities with the largest 2030 losses under the Maria-like hurricane and M7.5 earthquake scenarios.")


Saved FIG: phase3_outputs_v5/phase3_v5_top_municipal_losses_compare.png


In [37]:

# ----------------------------
# Chart 17: labeled regional scenario maps (2030)
# ----------------------------
import unicodedata
import json
from matplotlib.patches import Polygon as MplPolygon
from matplotlib.collections import PatchCollection

def _normalize_name(x):
    if pd.isna(x):
        return None
    x = str(x).strip()
    x = unicodedata.normalize("NFKD", x).encode("ascii", "ignore").decode("utf-8")
    x = " ".join(x.split())
    return x.lower()

def _find_phase3_geojson():
    candidates = [
        Path("pr_municipalities.geojson"),
        Path("data") / "pr_municipalities.geojson",
        Path("..") / "data" / "pr_municipalities.geojson",
        Path("..") / "data" / "tl_2022_72_cousub" / "pr_municipalities.geojson",
        OUTPUT_DIR.parent / "pr_municipalities.geojson",
        OUTPUT_DIR.parent / "data" / "pr_municipalities.geojson",
        OUTPUT_DIR.parent / "data" / "tl_2022_72_cousub" / "pr_municipalities.geojson",
    ]
    for p in candidates:
        if p.exists():
            return p.resolve()
    return None

def _detect_geo_name_col(columns):
    for candidate in ["municipio", "MUNICIPIO", "name", "NAME", "municipality", "Municipality", "NAMELSAD", "county_subdivision"]:
        if candidate in columns:
            return candidate
    return None

def _detect_geo_name_from_properties(props):
    for candidate in ["municipio", "MUNICIPIO", "name", "NAME", "municipality", "Municipality", "NAMELSAD", "county_subdivision"]:
        if candidate in props and props[candidate]:
            return candidate
    return None

def _make_polygon_patches(geometry):
    patches = []
    if not geometry:
        return patches
    gtype = geometry.get("type")
    coords = geometry.get("coordinates", [])
    if gtype == "Polygon":
        polys = [coords]
    elif gtype == "MultiPolygon":
        polys = coords
    else:
        return patches
    for poly in polys:
        if not poly:
            continue
        outer = poly[0]
        if len(outer) >= 3:
            patches.append(MplPolygon(outer, closed=True))
    return patches

def _approx_label_point(geometry):
    if not geometry:
        return None
    pts = []
    gtype = geometry.get("type")
    coords = geometry.get("coordinates", [])
    if gtype == "Polygon":
        polys = [coords]
    elif gtype == "MultiPolygon":
        polys = coords
    else:
        return None
    for poly in polys:
        if poly and poly[0]:
            pts.extend(poly[0])
    if not pts:
        return None
    xs = [p[0] for p in pts]
    ys = [p[1] for p in pts]
    return (sum(xs) / len(xs), sum(ys) / len(ys))

geo_path = _find_phase3_geojson()
if geo_path is None:
    print("Regional scenario maps skipped because no Puerto Rico municipality GeoJSON was found.")
else:
    print(f"Using geometry file for Phase 3 regional maps: {geo_path}")
    phase3_region_map_saved = []
    muni_region_lookup = (
        df[["municipio", "region"]]
        .dropna()
        .drop_duplicates("municipio")
        .assign(municipio_norm=lambda d: d["municipio"].map(_normalize_name))
    )

    try:
        import geopandas as gpd

        geo_muni = gpd.read_file(geo_path)
        geo_name_col = _detect_geo_name_col(list(geo_muni.columns))
        if geo_name_col is None:
            raise ValueError("Could not detect municipality name column in geometry file.")

        geo_muni["municipio_norm"] = geo_muni[geo_name_col].map(_normalize_name)
        geo_muni = geo_muni.merge(
            muni_region_lookup[["municipio_norm", "region"]],
            on="municipio_norm",
            how="left",
        )
        geo_muni["region"] = geo_muni["region"].fillna("Unknown")
        region_shapes = geo_muni[geo_muni["region"].notna()].dissolve(by="region").reset_index()
        map_backend = "geopandas"

    except Exception as e:
        print(f"geopandas region-map path failed ({e}); falling back to raw GeoJSON plotting.")
        with open(geo_path, "r", encoding="utf-8") as f:
            geojson_obj = json.load(f)

        records = []
        geo_name_col = None
        for feat in geojson_obj.get("features", []):
            props = feat.get("properties", {}) or {}
            if geo_name_col is None:
                geo_name_col = _detect_geo_name_from_properties(props)
            muni_name = props.get(geo_name_col) if geo_name_col is not None else None
            records.append({
                "municipio_norm": _normalize_name(muni_name) if muni_name is not None else None,
                "geometry": feat.get("geometry"),
            })

        map_df = pd.DataFrame(records).merge(
            muni_region_lookup[["municipio_norm", "region"]],
            on="municipio_norm",
            how="left",
        )
        map_df["region"] = map_df["region"].fillna("Unknown")
        region_shapes = map_df.copy()
        map_backend = "geojson_fallback"

    focus_map_scenarios = [
        s for s in ["baseline", "hurricane_cat3", "hurricane_cat5_maria_like", "earthquake_m6_8", "earthquake_m7_5"]
        if s in region_2030["scenario"].unique()
    ]
    if not focus_map_scenarios:
        focus_map_scenarios = list(region_2030["scenario"].dropna().unique())

    for scen in focus_map_scenarios:
        scen_df = region_2030[region_2030["scenario"] == scen].copy()
        if scen_df.empty:
            continue

        if map_backend == "geopandas":
            plot_gdf = region_shapes.merge(
                scen_df[["region", "pct_impact_vs_baseline", "abs_impact_vs_baseline", "p50", "baseline_p50"]],
                on="region",
                how="left",
            )

            fig, ax = plt.subplots(figsize=(12, 6))
            plot_gdf.plot(
                column="pct_impact_vs_baseline",
                cmap="RdYlBu",
                linewidth=1.0,
                edgecolor="black",
                legend=True,
                ax=ax,
                missing_kwds={"color": "#d9d9d9", "label": "Missing"},
            )

            label_points = plot_gdf.geometry.representative_point()
            for (_, row), pt in zip(plot_gdf.iterrows(), label_points):
                if pt is not None and not pt.is_empty and pd.notna(row.get("pct_impact_vs_baseline")):
                    ax.text(
                        pt.x,
                        pt.y,
                        f"{row['region']}\n{row['pct_impact_vs_baseline']:.2f}%",
                        fontsize=9,
                        ha="center",
                        va="center",
                        bbox=dict(boxstyle="round,pad=0.2", fc="white", ec="none", alpha=0.8),
                    )

            scen_title = scen.replace("_", " ").title()
            ax.set_title(f"Regional 2030 Population Impact vs Baseline — {scen_title}", fontsize=16)
            ax.axis("off")

            fname = f"phase3_v5_map_region_2030_pct_impact_{scen}.png"
            saved_files.append(save_fig(fig, fname))
            add_chart_record(fname, f"Regional 2030 impact map — {scen_title}", f"Labeled Puerto Rico regional map showing 2030 percent population impact versus baseline for the {scen_title} scenario.")
            phase3_region_map_saved.append(fname)

        else:
            fig, ax = plt.subplots(figsize=(12, 6))
            vals = scen_df.set_index("region")["pct_impact_vs_baseline"].to_dict()

            patches = []
            colors = []
            label_pts = {}
            from matplotlib import cm, colors as mcolors
            finite_vals = [v for v in vals.values() if pd.notna(v)]
            if finite_vals:
                norm = mcolors.Normalize(vmin=min(finite_vals), vmax=max(finite_vals))
            else:
                norm = mcolors.Normalize(vmin=-1, vmax=1)
            cmap = cm.get_cmap("RdYlBu")

            for _, row in region_shapes.iterrows():
                region_name = row["region"]
                geom = row["geometry"]
                region_val = vals.get(region_name, np.nan)
                poly_patches = _make_polygon_patches(geom)
                if not poly_patches:
                    continue
                patches.extend(poly_patches)
                facecolor = cmap(norm(region_val)) if pd.notna(region_val) else (0.85, 0.85, 0.85, 1.0)
                colors.extend([facecolor] * len(poly_patches))
                if region_name not in label_pts:
                    label_pts[region_name] = _approx_label_point(geom)

            pc = PatchCollection(patches, facecolor=colors, edgecolor="black", linewidth=1.0)
            ax.add_collection(pc)
            ax.autoscale_view()
            ax.set_aspect("equal")
            ax.axis("off")

            for region_name, pt in label_pts.items():
                region_val = vals.get(region_name, np.nan)
                if pt is not None and pd.notna(region_val):
                    ax.text(
                        pt[0],
                        pt[1],
                        f"{region_name}\n{region_val:.2f}%",
                        fontsize=9,
                        ha="center",
                        va="center",
                        bbox=dict(boxstyle="round,pad=0.2", fc="white", ec="none", alpha=0.8),
                    )

            scen_title = scen.replace("_", " ").title()
            ax.set_title(f"Regional 2030 Population Impact vs Baseline — {scen_title}", fontsize=16)
            sm = plt.cm.ScalarMappable(cmap="RdYlBu", norm=norm)
            sm.set_array([])
            fig.colorbar(sm, ax=ax, fraction=0.03, pad=0.02, label="% difference vs baseline")

            fname = f"phase3_v5_map_region_2030_pct_impact_{scen}.png"
            saved_files.append(save_fig(fig, fname))
            add_chart_record(fname, f"Regional 2030 impact map — {scen_title}", f"Labeled Puerto Rico regional map showing 2030 percent population impact versus baseline for the {scen_title} scenario.")
            phase3_region_map_saved.append(fname)

    print("Saved Phase 3 regional scenario maps:", phase3_region_map_saved)


Using geometry file for Phase 3 regional maps: /Users/andreruiz/Documents/Capstone/CAPSTONE_DATA_SCIENCE_RIVERARUIZ/data/tl_2022_72_cousub/pr_municipalities.geojson
geopandas region-map path failed (No module named 'geopandas'); falling back to raw GeoJSON plotting.
Saved FIG: phase3_outputs_v5/phase3_v5_map_region_2030_pct_impact_baseline.png
Saved FIG: phase3_outputs_v5/phase3_v5_map_region_2030_pct_impact_hurricane_cat3.png
Saved FIG: phase3_outputs_v5/phase3_v5_map_region_2030_pct_impact_hurricane_cat5_maria_like.png
Saved FIG: phase3_outputs_v5/phase3_v5_map_region_2030_pct_impact_earthquake_m6_8.png
Saved FIG: phase3_outputs_v5/phase3_v5_map_region_2030_pct_impact_earthquake_m7_5.png
Saved Phase 3 regional scenario maps: ['phase3_v5_map_region_2030_pct_impact_baseline.png', 'phase3_v5_map_region_2030_pct_impact_hurricane_cat3.png', 'phase3_v5_map_region_2030_pct_impact_hurricane_cat5_maria_like.png', 'phase3_v5_map_region_2030_pct_impact_earthquake_m6_8.png', 'phase3_v5_map_regio

## Conclusion

This Phase 3 notebook extends the finalized Phase 2 modeling framework into a **multi-scale scenario forecasting system** for Puerto Rico. The forecasting engine is anchored in the **final tuned Lasso model** selected in Phase 2 and applies stylized hurricane and earthquake shocks to generate municipal, regional, and island-wide projections through 2030.

---

### Backbone carried forward from Phase 2

- **Final model:** Lasso  
- **Confirmed Phase 2 test R²:** **0.8678**  
- **Confirmed Phase 2 test RMSE:** **0.5468**  
- **Design:** chronological **80/20 train–test split**, with a validation year embedded within the training block  

---

### Why this matters

Maintaining the same model backbone ensures **methodological consistency across phases**. The scenario forecasts are therefore grounded in the **best-performing out-of-sample model**, rather than relying on a different or weaker specification. This strengthens the credibility of the results and directly links the forecasting outputs to the validated predictive structure established in Phase 2.

---

### How to interpret the results

- The **baseline trajectory** represents the conditional forecast generated by the Lasso model using lagged, rolling, structural, and geographic predictors.  
- The **hurricane and earthquake scenarios** act as **stress tests**, introducing shocks into the same feature space used by the model while incorporating event-year adjustments.  
- **Municipal-level forecasts** exhibit higher variability due to local heterogeneity, while **regional and island-level summaries** provide more stable and interpretable patterns.  
- These projections should be understood as **scenario-based, forward-looking simulations**, not causal estimates of disaster impacts.  

---

### Model interpretation insight

The strong performance of the Lasso model suggests that **regularized linear structure combined with temporally lagged and smoothed predictors** is well-suited to capturing Puerto Rico’s population dynamics. Compared to tree-based models, Lasso likely benefits from:

- effective **regularization**, reducing overfitting in a moderate sample setting  
- the ability to leverage **structured lag relationships** directly  
- greater **stability in out-of-sample prediction**  

---

### Key takeaway

By carrying forward the tuned Lasso model into Phase 3, the forecasting framework remains fully aligned with the strongest validated model from Phase 2. This integration enables a coherent transition from model evaluation to **policy-relevant scenario analysis**, where the resulting projections provide a structured way to assess how different disaster intensities may influence Puerto Rico’s population trajectory over time.

In [38]:

# ----------------------------
# Chart index / manifest
# ----------------------------
chart_index_df = pd.DataFrame(chart_index_records)
manifest_df = pd.DataFrame({"filepath": saved_files})

saved_files.append(save_csv(chart_index_df, "phase3_v5_chart_index.csv"))
saved_files.append(save_csv(manifest_df, "phase3_v5_saved_files_manifest.csv"))

chart_index_df.head()


Saved CSV: phase3_outputs_v5/phase3_v5_chart_index.csv
Saved CSV: phase3_outputs_v5/phase3_v5_saved_files_manifest.csv


,filename,title,description
0,phase3_v5_island_fan_charts_all_scenarios.png,Island fan charts by scenario,Observed island history with forecast fan char...
1,phase3_v5_island_hurricane_ladder.png,Island hurricane ladder,Median island forecast paths comparing baselin...
2,phase3_v5_island_earthquake_ladder.png,Island earthquake ladder,Median island forecast paths comparing baselin...
3,phase3_v5_side_by_side_baseline_vs_hazards.png,Baseline vs selected hazards,Side-by-side island-level comparison panels fo...
4,phase3_v5_island_impact_over_time.png,Island impact over time,Difference between scenario median and baselin...
